In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.firmable_v2_bronze"
SILVER = "workspace.firmable_v2_silver"

# ============================================================
# 1. READ BRONZE INPUTS
# ============================================================

raw = spark.table(f"{BRONZE}.employees_ingestion_raw")
companies = spark.table(f"{BRONZE}.companies")

# ============================================================
# 2. NORMALIZE EMPLOYEE ATTRIBUTES
# ============================================================

df = (
    raw
    .withColumn(
        "job_title_clean",
        F.when(
            F.col("job_title").isNull() |
            (F.trim(F.col("job_title")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.initcap(
                F.regexp_replace(F.trim(F.col("job_title")), r"\s+", " ")
            )
        )
    )
    .withColumn(
        "department_clean",
        F.when(
            F.col("department").isNull() |
            (F.trim(F.col("department")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.initcap(
                F.regexp_replace(
                    F.regexp_replace(
                        F.trim(F.col("department")),
                        r"\s+",
                        " "
                    ),
                    r"(?i)\s+and\s+",
                    " & "
                )
            )
        )
    )
    .withColumn(
        "business_unit_clean",
        F.when(
            F.col("business_unit").isNull() |
            (F.trim(F.col("business_unit")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.initcap(F.trim(F.col("business_unit")))
        )
    )
    .withColumn("team_id_clean", F.col("team_id"))
    .withColumn(
        "team_name_clean",
        F.when(
            F.col("team_name").isNull() |
            (F.trim(F.col("team_name")) == ""),
            F.lit(None).cast("string")
        ).otherwise(F.initcap(F.trim(F.col("team_name"))))
    )
)

# Restore common acronyms after title casing.
for acronym in ["CEO", "CTO", "CFO", "COO", "CRO", "CHRO", "CIO", "IT", "QA"]:
    df = df.withColumn(
        "job_title_clean",
        F.regexp_replace(
            F.col("job_title_clean"),
            rf"(?i)\b{acronym.title()}\b",
            acronym
        )
    )

# ============================================================
# 3. TITLE CLASSIFICATION RULES
# ============================================================

title = F.coalesce(F.col("job_title_clean"), F.lit(""))

# Some titles contain the word Manager but do not reliably
# establish people-management responsibility.
ambiguous_ic = title.rlike(
    r"(?i)^(?:(?:Senior|Junior)\s+)?"
    r"(?:Account Manager|Key Account Manager|"
    r"Client Account Manager|HR Business Partner)$"
)

# Recognized individual-contributor title families.
known_ic = title.rlike(
    r"(?i).*(Engineer|Analyst|Scientist|Specialist|Associate|"
    r"Executive|Coordinator|Representative|Consultant|Counsel|"
    r"Business Partner|Account Manager|Architect|Designer|"
    r"Recruiter|Accountant|Auditor|Developer|Researcher|"
    r"Technician|Administrator|Support|Operator|"
    r"Marketing|Sales).*"
)

# ============================================================
# 4. INFER SENIORITY
# Lower number = more senior.
# Unrecognized titles remain NULL.
# ============================================================

df = df.withColumn(
    "inferred_seniority_level",
    F.when(title.rlike(r"(?i)^Chief Executive Officer$"), 0)
     .when(title.rlike(r"(?i)^Chief .* Officer$"), 1)
     .when(title.rlike(r"(?i)^President\s*-\s*.*Division$"), 2)
     .when(title.rlike(r"(?i)^Senior Vice President"), 3)
     .when(title.rlike(r"(?i)^Vice President"), 4)
     .when(title.rlike(r"(?i)^(Head|Department Head) of "), 5)
     .when(title.rlike(r"(?i)^Senior Director"), 6)
     .when(title.rlike(r"(?i)^Director"), 7)
     .when(
         title.rlike(r"(?i)^Senior .* Manager$") & ~ambiguous_ic,
         8
     )
     .when(
         title.rlike(r"(?i)^.* Manager$") & ~ambiguous_ic,
         9
     )
     .when(title.rlike(r"(?i).*Lead.*"), 10)
     .when(title.rlike(r"(?i)^Senior ") & (known_ic | ambiguous_ic), 11)
     .when(title.rlike(r"(?i)^Junior ") & (known_ic | ambiguous_ic), 13)
     .when(known_ic | ambiguous_ic, 12)
     .otherwise(F.lit(None).cast("int"))
)

# ============================================================
# 5. INFER MANAGEMENT AUTHORITY
# Supporting evidence only; not a manager assignment.
# ============================================================

df = df.withColumn(
    "management_authority",
    F.when(
        F.col("inferred_seniority_level").between(0, 2),
        "Executive"
    )
    .when(
        F.col("inferred_seniority_level").between(3, 9),
        "People Manager"
    )
    .when(
        F.col("inferred_seniority_level") == 10,
        "Potential People Manager"
    )
    .when(
        F.col("inferred_seniority_level").between(11, 13),
        "Individual Contributor"
    )
    .otherwise("Unknown")
)

# ============================================================
# 6. INFER TITLE FUNCTION
# Department is used as a fallback, not proof of actual function.
# ============================================================

df = df.withColumn(
    "title_function",
    F.when(title.rlike(r"(?i)^Chief Executive Officer$"), "Executive")
     .when(title.rlike(r"(?i)^Chief Technology Officer$"), "Technology")
     .when(title.rlike(r"(?i)^Chief Financial Officer$"), "Finance")
     .when(title.rlike(r"(?i)^Chief Operating Officer$"), "Operations")
     .when(title.rlike(r"(?i)^Chief Revenue Officer$"), "Sales")
     .when(title.rlike(r"(?i)^Chief Human Resources Officer$"), "Human Resources")
     .when(title.rlike(r"(?i)^Chief Information Officer$"), "Technology")
     .otherwise(F.col("department_clean"))
)

# ============================================================
# 7. JOIN COMPANY CONTEXT
# ============================================================

# ============================================================
# 7. JOIN COMPANY CONTEXT
# ============================================================

company_context = companies.select(
    "company_id",
    "company_name",
    "industry",
    "company_size_category",
    "employee_count",
    "org_structure"
)

df = df.join(
    company_context,
    on="company_id",
    how="left"
)

# ============================================================
# 8. DATA QUALITY FLAGS
# ============================================================

df = (
    df
    .withColumn("missing_title_flag", F.col("job_title_clean").isNull())
    .withColumn("missing_department_flag", F.col("department_clean").isNull())
    .withColumn("missing_business_unit_flag", F.col("business_unit_clean").isNull())
    .withColumn("missing_team_flag", F.col("team_id_clean").isNull())
    .withColumn(
        "missing_seniority_flag",
        F.col("inferred_seniority_level").isNull()
    )
    .withColumn(
        "title_normalization_flag",
        F.col("job_title").isNotNull() &
        (F.trim(F.col("job_title")) != F.col("job_title_clean"))
    )
    .withColumn(
        "ambiguous_title_flag",
        ambiguous_ic
    )
)

# ============================================================
# 9. PUBLISH SILVER TABLES
# ============================================================

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.employees_clean")
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.employees_title_normalized")
)

# ============================================================
# 10. CONSOLIDATED QUALITY SUMMARY
# ============================================================

summary = df.agg(
    F.count("*").alias("total_employees"),
    F.countDistinct("employee_id").alias("unique_employees"),
    F.sum(F.col("missing_title_flag").cast("int")).alias("missing_titles"),
    F.sum(F.col("missing_department_flag").cast("int")).alias("missing_departments"),
    F.sum(F.col("missing_seniority_flag").cast("int")).alias("unknown_seniority"),
    F.sum(F.col("ambiguous_title_flag").cast("int")).alias("ambiguous_titles"),
    F.sum(F.col("missing_team_flag").cast("int")).alias("employees_without_team")
)

print("SILVER TRANSFORMATION COMPLETED")
summary.show(truncate=False)

print("Management authority distribution:")
df.groupBy("management_authority").count().orderBy("management_authority").show()

print("Seniority distribution:")
df.groupBy("inferred_seniority_level").count().orderBy("inferred_seniority_level").show()

# Essential integrity checks
total = df.count()
unique_ids = df.select("employee_id").distinct().count()
missing_company = df.filter(F.col("org_structure").isNull()).count()

assert total == unique_ids, "Duplicate employee IDs detected."
assert missing_company == 0, "Some employees have no matching company metadata."

print("All Silver integrity checks passed.")

In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.firmable_v2_silver"

# Read the current Silver table
df = spark.table(f"{SILVER}.employees_clean")

# --------------------------------------------------
# 1. Normalize Sr. Director to Senior Director
# --------------------------------------------------

df = df.withColumn(
    "job_title_clean",
    F.regexp_replace(
        F.col("job_title_clean"),
        r"(?i)^Sr\.?\s+Director",
        "Senior Director"
    )
)

title = F.coalesce(
    F.col("job_title_clean"),
    F.lit("")
)

# --------------------------------------------------
# 2. Correct seniority for affected titles
# --------------------------------------------------

df = df.withColumn(
    "inferred_seniority_level",
    F.when(
        title.rlike(r"(?i)^Senior Director"),
        6
    )
    .when(
        title.rlike(r"(?i)^Product Owner$|^Content Strategist$"),
        12
    )
    .otherwise(F.col("inferred_seniority_level"))
)

# --------------------------------------------------
# 3. Recalculate management authority
# --------------------------------------------------

df = df.withColumn(
    "management_authority",
    F.when(
        F.col("inferred_seniority_level").between(0, 2),
        "Executive"
    )
    .when(
        F.col("inferred_seniority_level").between(3, 9),
        "People Manager"
    )
    .when(
        F.col("inferred_seniority_level") == 10,
        "Potential People Manager"
    )
    .when(
        F.col("inferred_seniority_level").between(11, 13),
        "Individual Contributor"
    )
    .otherwise("Unknown")
)

# --------------------------------------------------
# 4. Update data-quality flags
# --------------------------------------------------

df = df.withColumn(
    "missing_seniority_flag",
    F.col("inferred_seniority_level").isNull()
)

df = df.withColumn(
    "ambiguous_title_flag",
    title.rlike(
        r"(?i)^(?:(?:Senior|Junior)\s+)?"
        r"(?:Account Manager|Key Account Manager|"
        r"Client Account Manager|HR Business Partner)$"
    )
)

# --------------------------------------------------
# 5. Update both Silver tables
# --------------------------------------------------

for table_name in [
    "employees_clean",
    "employees_title_normalized"
]:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.{table_name}")
    )

# --------------------------------------------------
# 6. Final classification checks
# --------------------------------------------------

print("SILVER TITLE CORRECTION COMPLETED")

df.groupBy(
    "inferred_seniority_level",
    "management_authority"
).count().orderBy(
    "inferred_seniority_level"
).show(truncate=False)

print("Corrected title samples:")

df.filter(
    F.col("job_title_clean").rlike(
        r"(?i)^Senior Director|^Product Owner$|^Content Strategist$"
    )
).groupBy(
    "job_title_clean",
    "inferred_seniority_level",
    "management_authority"
).count().show(30, truncate=False)

print(
    "Remaining unknown seniority:",
    df.filter(
        F.col("inferred_seniority_level").isNull()
    ).count()
)